In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/llm-agentic-legal-information-retrieval/sample_submission.csv
/kaggle/input/llm-agentic-legal-information-retrieval/laws_de.csv
/kaggle/input/llm-agentic-legal-information-retrieval/val.csv
/kaggle/input/llm-agentic-legal-information-retrieval/court_considerations.csv
/kaggle/input/llm-agentic-legal-information-retrieval/train.csv
/kaggle/input/llm-agentic-legal-information-retrieval/test.csv


In [2]:
"""
=============================================================================
EDA: LLM Agentic Legal IR Für Swiss Law
=============================================================================
Understand the data before building a IR pipeline.
"""

import pandas as pd
import numpy as np
import re
from collections import Counter
from pathlib import Path

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

BASE = Path("/kaggle/input/llm-agentic-legal-information-retrieval")

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD ALL FILES
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print("1. LOADING DATA FILES")
print("=" * 80)

train = pd.read_csv(BASE / "train.csv")
val = pd.read_csv(BASE / "val.csv")
test = pd.read_csv(BASE / "test.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
laws = pd.read_csv(BASE / "laws_de.csv")

# Court considerations is large – load with progress info
print("Loading court_considerations.csv (this may take a while)...")
court = pd.read_csv(BASE / "court_considerations.csv")

for name, df in [("train", train), ("val", val), ("test", test),
                  ("sample_submission", sample_sub),
                  ("laws_de", laws), ("court_considerations", court)]:
    print(f"  {name:25s} → {df.shape[0]:>9,} rows × {df.shape[1]} cols  "
          f"| mem: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")

# ─────────────────────────────────────────────────────────────────────────────
# 2. TRAIN SET EXPLORATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("2. TRAIN SET")
print("=" * 80)

print(f"\nColumns: {list(train.columns)}")
print(f"Shape: {train.shape}")
print(f"\nFirst 5 rows:")
print(train.head())

print(f"\nNull counts:\n{train.isnull().sum()}")

# Query language analysis
print(f"\n--- Query Stats ---")
train["query_len"] = train["query"].str.len()
train["query_word_count"] = train["query"].str.split().str.len()
print(f"Query length (chars):  mean={train['query_len'].mean():.0f}, "
      f"median={train['query_len'].median():.0f}, "
      f"min={train['query_len'].min()}, max={train['query_len'].max()}")
print(f"Query word count:      mean={train['query_word_count'].mean():.1f}, "
      f"median={train['query_word_count'].median():.0f}, "
      f"min={train['query_word_count'].min()}, max={train['query_word_count'].max()}")

# Detect query languages (simple heuristic)
def detect_lang_heuristic(text):
    text_lower = text.lower()
    de_words = {"der", "die", "das", "und", "ist", "ein", "für", "von", "mit", "auf", "nicht", "sich", "des", "dem", "den"}
    fr_words = {"le", "la", "les", "des", "est", "une", "dans", "pour", "que", "qui", "pas", "sur", "avec"}
    en_words = {"the", "is", "are", "was", "and", "for", "that", "with", "this", "from", "which", "what", "how"}
    it_words = {"il", "la", "che", "di", "del", "della", "sono", "per", "con", "una"}
    
    words = set(text_lower.split())
    scores = {
        "DE": len(words & de_words),
        "FR": len(words & fr_words),
        "EN": len(words & en_words),
        "IT": len(words & it_words),
    }
    return max(scores, key=scores.get)

train["query_lang"] = train["query"].apply(detect_lang_heuristic)
print(f"\nQuery language distribution (heuristic):")
print(train["query_lang"].value_counts())

# Citation analysis
print(f"\n--- Citation Stats (train) ---")
train["citations_list"] = train["gold_citations"].str.split(";")
train["num_citations"] = train["citations_list"].str.len()
print(f"Citations per query:   mean={train['num_citations'].mean():.1f}, "
      f"median={train['num_citations'].median():.0f}, "
      f"min={train['num_citations'].min()}, max={train['num_citations'].max()}")
print(f"\nDistribution of citation counts:")
print(train["num_citations"].value_counts().sort_index().head(30))

# Flatten all train citations
all_train_cits = [c.strip() for cits in train["citations_list"] for c in cits]
print(f"\nTotal citation instances in train: {len(all_train_cits)}")
print(f"Unique citations in train: {len(set(all_train_cits))}")

# Citation type classification
def classify_citation(cit):
    cit = cit.strip()
    if cit.startswith("BGE"):
        return "BGE (leading decision)"
    elif re.match(r"\d+[A-Z]_", cit):
        return "Docket (non-leading decision)"
    elif cit.startswith("Art."):
        return "Article (law)"
    else:
        return "Other"

train_cit_types = [classify_citation(c) for c in all_train_cits]
print(f"\nCitation type distribution (train):")
for ctype, count in Counter(train_cit_types).most_common():
    print(f"  {ctype:40s} {count:>6}  ({100*count/len(train_cit_types):.1f}%)")

# Most common citations
print(f"\nTop 20 most cited references in train:")
cit_counter = Counter(all_train_cits)
for cit, count in cit_counter.most_common(20):
    print(f"  {count:>4}x  {cit}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. VALIDATION SET EXPLORATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("3. VALIDATION SET")
print("=" * 80)

print(f"\nShape: {val.shape}")
print(f"Columns: {list(val.columns)}")
print(f"\nAll val queries:")
for i, row in val.iterrows():
    print(f"\n  [{row['query_id']}]")
    print(f"  Q: {row['query'][:200]}")
    cits = row['gold_citations'].split(";")
    print(f"  Citations ({len(cits)}): {row['gold_citations'][:200]}...")

val["citations_list"] = val["gold_citations"].str.split(";")
val["num_citations"] = val["citations_list"].str.len()
print(f"\nVal citations per query: mean={val['num_citations'].mean():.1f}, "
      f"median={val['num_citations'].median():.0f}, "
      f"min={val['num_citations'].min()}, max={val['num_citations'].max()}")

# Check overlap between val and train citations
val_cits = set(c.strip() for cits in val["citations_list"] for c in cits)
train_cits = set(all_train_cits)
overlap = val_cits & train_cits
print(f"\nUnique val citations: {len(val_cits)}")
print(f"Val citations also in train: {len(overlap)} ({100*len(overlap)/len(val_cits):.1f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# 4. TEST SET EXPLORATION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("4. TEST SET")
print("=" * 80)

print(f"\nShape: {test.shape}")
print(f"Columns: {list(test.columns)}")
print(f"\nAll test queries:")
for i, row in test.iterrows():
    print(f"\n  [{row['query_id']}] {row['query'][:250]}")

test["query_len"] = test["query"].str.len()
test["query_word_count"] = test["query"].str.split().str.len()
print(f"\nTest query length (chars): mean={test['query_len'].mean():.0f}, "
      f"median={test['query_len'].median():.0f}")
print(f"Test query words:          mean={test['query_word_count'].mean():.1f}, "
      f"median={test['query_word_count'].median():.0f}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. SAMPLE SUBMISSION
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("5. SAMPLE SUBMISSION")
print("=" * 80)
print(f"Shape: {sample_sub.shape}")
print(f"Columns: {list(sample_sub.columns)}")
print(sample_sub.head(10))

# ─────────────────────────────────────────────────────────────────────────────
# 6. LAWS CORPUS (laws_de.csv)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("6. LAWS CORPUS (laws_de.csv)")
print("=" * 80)

print(f"\nShape: {laws.shape}")
print(f"Columns: {list(laws.columns)}")
print(f"Nulls:\n{laws.isnull().sum()}")

laws["text_len"] = laws["text"].str.len()
laws["text_words"] = laws["text"].str.split().str.len()
print(f"\nText length (chars): mean={laws['text_len'].mean():.0f}, "
      f"median={laws['text_len'].median():.0f}, "
      f"min={laws['text_len'].min()}, max={laws['text_len'].max()}")
print(f"Text words:          mean={laws['text_words'].mean():.1f}, "
      f"median={laws['text_words'].median():.0f}")

# Extract law abbreviations
def extract_law_abbrev(citation):
    # e.g. "Art. 11 Abs. 2 OR" → "OR"
    parts = citation.strip().split()
    # The abbreviation is typically the last token
    if parts:
        return parts[-1]
    return "UNKNOWN"

laws["law_abbrev"] = laws["citation"].apply(extract_law_abbrev)
print(f"\nNumber of distinct law abbreviations: {laws['law_abbrev'].nunique()}")
print(f"\nTop 30 laws by article count:")
law_counts = laws["law_abbrev"].value_counts()
for abbrev, count in law_counts.head(30).items():
    print(f"  {abbrev:20s} {count:>6} articles")

# Check how many train citations are in the laws corpus
laws_citation_set = set(laws["citation"].values)
train_law_cits = [c for c in all_train_cits if classify_citation(c) == "Article (law)"]
train_law_in_corpus = [c for c in train_law_cits if c in laws_citation_set]
print(f"\nTrain law citations: {len(set(train_law_cits))} unique")
print(f"Found in laws corpus: {len(set(train_law_in_corpus))} "
      f"({100*len(set(train_law_in_corpus))/max(len(set(train_law_cits)),1):.1f}%)")

# Show mismatches (citations NOT found in corpus)
train_law_missing = set(train_law_cits) - laws_citation_set
if train_law_missing:
    print(f"\nTrain law citations NOT in corpus ({len(train_law_missing)}):")
    for c in sorted(train_law_missing)[:20]:
        print(f"  {c}")

# Sample law entries
print(f"\nSample law entries:")
for _, row in laws.sample(5, random_state=42).iterrows():
    print(f"\n  Citation: {row['citation']}")
    print(f"  Text: {row['text'][:300]}...")

# ─────────────────────────────────────────────────────────────────────────────
# 7. COURT CONSIDERATIONS CORPUS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("7. COURT CONSIDERATIONS CORPUS")
print("=" * 80)

print(f"\nShape: {court.shape}")
print(f"Columns: {list(court.columns)}")
print(f"Nulls:\n{court.isnull().sum()}")

court["text_len"] = court["text"].str.len()
court["text_words"] = court["text"].str.split().str.len()
print(f"\nText length (chars): mean={court['text_len'].mean():.0f}, "
      f"median={court['text_len'].median():.0f}, "
      f"min={court['text_len'].min()}, max={court['text_len'].max()}")
print(f"Text words:          mean={court['text_words'].mean():.1f}, "
      f"median={court['text_words'].median():.0f}")

# Classify court citations
def classify_court_citation(cit):
    if cit.startswith("BGE"):
        return "BGE"
    elif re.match(r"\d+[A-Z]_", cit):
        return "Docket"
    else:
        return "Other"

court["cit_type"] = court["citation"].apply(classify_court_citation)
print(f"\nCourt citation types:")
print(court["cit_type"].value_counts())

# Language detection on court texts (sample)
court_sample = court.sample(min(5000, len(court)), random_state=42)
court_sample["text_lang"] = court_sample["text"].apply(detect_lang_heuristic)
print(f"\nCourt text language distribution (sampled {len(court_sample)}):")
print(court_sample["text_lang"].value_counts())

# Check how many train court citations are in the corpus
court_citation_set = set(court["citation"].values)
train_court_cits = [c for c in all_train_cits if classify_citation(c) in 
                    ("BGE (leading decision)", "Docket (non-leading decision)")]
train_court_in_corpus = [c for c in train_court_cits if c in court_citation_set]
print(f"\nTrain court citations: {len(set(train_court_cits))} unique")
print(f"Found in court corpus: {len(set(train_court_in_corpus))} "
      f"({100*len(set(train_court_in_corpus))/max(len(set(train_court_cits)),1):.1f}%)")

train_court_missing = set(train_court_cits) - court_citation_set
if train_court_missing:
    print(f"\nTrain court citations NOT in corpus ({len(train_court_missing)}):")
    for c in sorted(train_court_missing)[:20]:
        print(f"  {c}")

# Sample court entries
print(f"\nSample court entries:")
for _, row in court.sample(3, random_state=42).iterrows():
    print(f"\n  Citation: {row['citation']}")
    print(f"  Text: {row['text'][:300]}...")

# ─────────────────────────────────────────────────────────────────────────────
# 8. FULL CORPUS OVERVIEW
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("8. FULL CORPUS OVERVIEW")
print("=" * 80)

total_corpus = len(laws) + len(court)
print(f"\nTotal retrievable units:  {total_corpus:>10,}")
print(f"  Laws:                   {len(laws):>10,}  ({100*len(laws)/total_corpus:.1f}%)")
print(f"  Court considerations:   {len(court):>10,}  ({100*len(court)/total_corpus:.1f}%)")

# Check for duplicate citations across corpora
laws_cits = set(laws["citation"])
court_cits = set(court["citation"])
overlap_corpus = laws_cits & court_cits
print(f"\nCitation overlap between laws and court: {len(overlap_corpus)}")
if overlap_corpus:
    print(f"  Examples: {list(overlap_corpus)[:5]}")

# Overall train citation coverage
all_train_cits_unique = set(all_train_cits)
all_corpus_cits = laws_cits | court_cits
train_covered = all_train_cits_unique & all_corpus_cits
train_uncovered = all_train_cits_unique - all_corpus_cits
print(f"\nTrain citations coverage:")
print(f"  Unique train citations:    {len(all_train_cits_unique)}")
print(f"  Found in corpus:           {len(train_covered)} ({100*len(train_covered)/len(all_train_cits_unique):.1f}%)")
print(f"  NOT in corpus:             {len(train_uncovered)} ({100*len(train_uncovered)/len(all_train_cits_unique):.1f}%)")

if train_uncovered:
    print(f"\n  Sample uncovered citations:")
    for c in sorted(train_uncovered)[:30]:
        print(f"    {c}")

# Val citation coverage
val_cits_unique = set(c.strip() for cits in val["citations_list"] for c in cits)
val_covered = val_cits_unique & all_corpus_cits
val_uncovered = val_cits_unique - all_corpus_cits
print(f"\nVal citations coverage:")
print(f"  Unique val citations:      {len(val_cits_unique)}")
print(f"  Found in corpus:           {len(val_covered)} ({100*len(val_covered)/len(val_cits_unique):.1f}%)")
print(f"  NOT in corpus:             {len(val_uncovered)} ({100*len(val_uncovered)/len(val_cits_unique):.1f}%)")
if val_uncovered:
    print(f"  Uncovered val citations:")
    for c in sorted(val_uncovered):
        print(f"    {c}")

# ─────────────────────────────────────────────────────────────────────────────
# 9. QUERY SIMILARITY: TRAIN vs VAL vs TEST
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("9. QUERY COMPARISON: TRAIN vs VAL vs TEST")
print("=" * 80)

print(f"\nTrain queries: {len(train)} (languages: {dict(train['query_lang'].value_counts())})")
print(f"Val queries:   {len(val)} (English)")
print(f"Test queries:  {len(test)} (English)")

# Compare query lengths
print(f"\nQuery length comparison (words):")
print(f"  Train:  mean={train['query_word_count'].mean():.1f}, median={train['query_word_count'].median():.0f}")
print(f"  Test:   mean={test['query_word_count'].mean():.1f}, median={test['query_word_count'].median():.0f}")

# ─────────────────────────────────────────────────────────────────────────────
# 10. CITATION FORMAT PATTERNS (for matching/normalization)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("10. CITATION FORMAT PATTERNS")
print("=" * 80)

# Analyze law citation patterns
print("\nLaw citation patterns (first 20):")
for cit in laws["citation"].head(20):
    print(f"  {cit}")

print("\nBGE citation patterns (first 20):")
bge_cits = court[court["cit_type"] == "BGE"]["citation"].head(20)
for cit in bge_cits:
    print(f"  {cit}")

print("\nDocket citation patterns (first 20):")
docket_cits = court[court["cit_type"] == "Docket"]["citation"].head(20)
for cit in docket_cits:
    print(f"  {cit}")

# Check for trailing/leading whitespace issues
print("\n--- Whitespace / Formatting Issues ---")
laws_ws = laws["citation"].apply(lambda x: x != x.strip()).sum()
court_ws = court["citation"].apply(lambda x: x != x.strip()).sum()
print(f"Laws citations with whitespace issues: {laws_ws}")
print(f"Court citations with whitespace issues: {court_ws}")

# Check for period variations in E. vs E
print("\nBGE 'E.' vs 'E' pattern check (sample):")
bge_with_period = court[court["citation"].str.contains(r"E\.", na=False)].shape[0]
bge_without_period = court[court["citation"].str.contains(r"E \d", na=False)].shape[0]
print(f"  Citations with 'E.' : {bge_with_period}")
print(f"  Citations with 'E ' (no period): {bge_without_period}")

# ─────────────────────────────────────────────────────────────────────────────
# 11. KEY TAKEAWAYS SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("11. KEY TAKEAWAYS")
print("=" * 80)

print(f"""
SUMMARY
-------
• Train set: {len(train)} queries (mostly non-English), avg {train['num_citations'].mean():.1f} citations each
• Val set: {len(val)} queries (English), avg {val['num_citations'].mean():.1f} citations each  
• Test set: {len(test)} queries (English), need to predict citations
• Corpus: {len(laws):,} law articles + {len(court):,} court considerations = {total_corpus:,} total
• Citation coverage: {100*len(train_covered)/len(all_train_cits_unique):.1f}% of train citations found in corpus
• Val coverage: {100*len(val_covered)/len(val_cits_unique):.1f}% of val citations found in corpus
• Cross-lingual challenge: queries in EN, sources mostly in DE
• Citation types: laws (Art.), BGE (leading decisions), Docket (non-leading)
• Key challenge: exact citation string matching required

STRATEGY IMPLICATIONS
---------------------
1. Train set is multilingual → val/test are English → need cross-lingual approach
2. Many citations NOT in corpus → some gold answers may be unpredictable
3. Variable citation counts → system should be adaptive
4. Exact string matching → citation normalization is critical
5. Corpus is large ({total_corpus:,}) → need efficient retrieval (BM25 + embeddings)
6. Court corpus dominates → but laws may be more frequently cited
""")

print("EDA COMPLETE")

1. LOADING DATA FILES
Loading court_considerations.csv (this may take a while)...
  train                     →     1,139 rows × 3 cols  | mem: 2.8 MB
  val                       →        10 rows × 3 cols  | mem: 0.0 MB
  test                      →        40 rows × 2 cols  | mem: 0.1 MB
  sample_submission         →         2 rows × 2 cols  | mem: 0.0 MB
  laws_de                   →   175,933 rows × 3 cols  | mem: 105.4 MB
  court_considerations      → 2,476,315 rows × 2 cols  | mem: 2651.2 MB

2. TRAIN SET

Columns: ['query_id', 'query', 'gold_citations']
Shape: (1139, 3)

First 5 rows:
     query_id  \
0  train_0001   
1  train_0002   
2  train_0003   
3  train_0004   
4  train_0005   

                                                                                                                                                                                                     query  \
0  Die A AG betreibt seit den 1970er-Jahren auf der Parzelle Nr. yyy (Wohn- und Gewerbezone) e